In [ ]:
# ============================================================
# EXPERIMENT 9A: K-MEANS CLUSTERING FROM SCRATCH
# Synthetic Customer Dataset (Income vs Spending Score)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# Step 1 & 2: Create and Display Dataset
X = np.array([
    [2.0, 2.5], [2.5, 3.0], [3.0, 2.0], [3.5, 3.5],
    [7.0, 8.0], [7.5, 7.0], [8.0, 8.5], [8.5, 7.5],
    [5.0, 2.0], [5.5, 2.5], [6.0, 1.5], [6.5, 2.0]
])
print(f"Dataset\n================================\nIncome\tSpending Score\n" + "\n".join(f"{r[0]:.1f}\t{r[1]:.1f}" for r in X))

# Step 3 & 4: Set Clusters and Initialize Centroids
K = 3
centroids = np.array([X[0], X[4], X[8]])
print(f"\nInitial Centroids:\n{centroids}")

# Step 5, 6 & 7: Distance, Assign Clusters, and Update Centroids Functions
def assign_clusters(X_in, cents):
    return np.array([np.argmin([np.linalg.norm(p - c) for c in cents]) for p in X_in])

def update_centroids(X_in, clusts, k):
    return np.array([np.mean(X_in[clusts == i], axis=0) if len(X_in[clusts == i]) > 0 else X_in[np.random.randint(len(X_in))] for i in range(k)])

# Step 8: K-Means Algorithm
for iteration in range(100):
    clusters = assign_clusters(X, centroids)
    new_centroids = update_centroids(X, clusters, K)
    centroid_shift = np.linalg.norm(new_centroids - centroids)
    print(f"\nIteration {iteration + 1}\nCentroids:\n{new_centroids}\nCentroid Shift: {round(centroid_shift, 6)}")
    centroids = new_centroids
    if centroid_shift < 0.0001:
        print("\nK-Means Converged.")
        break

# Step 9 & 10: Final Assignment and Display Results
clusters = assign_clusters(X, centroids)
print("\n===================================\nFINAL K-MEANS RESULTS\n===================================")
for i in range(K):
    print(f"\nCluster {i + 1}\n{X[clusters == i]}")
print(f"\nFinal Centroids:\n{centroids}")

# Step 11: Calculate WCSS
wcss = sum(np.sum((X[clusters == k] - centroids[k])**2) for k in range(K))
print(f"\nWithin-Cluster Sum of Squares:\n{round(wcss, 4)}")

# Step 12: Predict Cluster for New Customer
new_customer = np.array([[6.2, 2.2]])
new_cluster = assign_clusters(new_customer, centroids)[0]
print(f"\n===================================\nNEW CUSTOMER PREDICTION\n===================================\nAnnual Income: {new_customer[0, 0]} Lakhs\nSpending Score: {new_customer[0, 1]}\nPredicted Cluster: {new_cluster + 1}")

# Step 13: Plot Clusters
plt.figure(figsize=(8, 6))
for k in range(K):
    plt.scatter(X[clusters == k, 0], X[clusters == k, 1], label=f"Cluster {k + 1}")
plt.scatter(centroids[:, 0], centroids[:, 1], marker="X", s=200, label="Centroids")
plt.scatter(new_customer[0, 0], new_customer[0, 1], marker="*", s=200, label="New Customer")
plt.xlabel("Annual Income (Lakhs)"), plt.ylabel("Spending Score"), plt.title("K-Means Clustering From Scratch")
plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 9B: DBSCAN CLUSTERING USING SCIKIT-LEARN
# Real-World Dataset: Iris Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
iris = load_iris()
X, target = iris.data, iris.target
feature_names, target_names = iris.feature_names, iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Actual Class"], df["Class Name"] = target, [target_names[i] for i in target]

print(f"Dataset Shape:\n{X.shape}")
print(f"\nFirst Five Records:\n{df.head()}")
print(f"\nFeature Names:\n{feature_names}\n\nDataset Description:\n{df.describe()}")

# Step 4-7: Feature Scaling, Apply DBSCAN, and Add Labels
X_scaled = StandardScaler().fit_transform(X)
cluster_labels = DBSCAN(eps=0.8, min_samples=5, metric="euclidean").fit_predict(X_scaled)
df["DBSCAN Cluster"] = cluster_labels
print(f"\nClustered Dataset:\n{df.head(10)}")

# Step 8-10: DBSCAN Results, Distribution, and Silhouette Score
unique_labels = set(cluster_labels)
n_clusters, n_noise = len(unique_labels - {-1}), np.sum(cluster_labels == -1)

print(f"\n===================================\nDBSCAN RESULTS\n===================================")
print(f"Number of Clusters: {n_clusters}\nNumber of Noise Points: {n_noise}")
print(f"\nCluster Distribution:\n{df['DBSCAN Cluster'].value_counts().sort_index()}")

non_noise = cluster_labels != -1
if n_clusters >= 2:
    print(f"\nSilhouette Score: {round(silhouette_score(X_scaled[non_noise], cluster_labels[non_noise]), 4)}")
else:
    print("\nSilhouette Score:\nCannot be calculated.")

# Step 11: Visualize DBSCAN Clusters
plt.figure(figsize=(9, 6))
for cluster in sorted(unique_labels):
    pts = cluster_labels == cluster
    plt.scatter(X[pts, 0], X[pts, 1], label="Noise" if cluster == -1 else f"Cluster {cluster}")
plt.xlabel("Sepal Length (cm)"), plt.ylabel("Sepal Width (cm)")
plt.title("DBSCAN Clustering - Iris Dataset"), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 12-13: Actual vs DBSCAN Cluster and Save Results
print(f"\nActual Class vs DBSCAN Cluster:\n{pd.DataFrame({'Actual Class': target, 'DBSCAN Cluster': cluster_labels}).head(20)}")
df.to_csv("Iris_DBSCAN_Results.csv", index=False)
print("\nResults saved as:\nIris_DBSCAN_Results.csv")

In [ ]:
# ============================================================
# EXPERIMENT 9C: HIERARCHICAL AGGLOMERATIVE CLUSTERING
# Real-World Dataset: Iris Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
iris = load_iris()
X, target = iris.data, iris.target
feature_names, target_names = iris.feature_names, iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Actual Class"], df["Class Name"] = target, [target_names[i] for i in target]

print(f"Dataset Shape:\n{X.shape}\n\nFirst Five Records:\n{df.head()}")
print(f"\nFeature Names:\n{feature_names}\n\nDataset Description:\n{df.describe()}")

# Step 4-5: Feature Scaling and Dendrogram
X_scaled = StandardScaler().fit_transform(X)
linkage_matrix = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(linkage_matrix, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=10)
plt.title("Hierarchical Clustering Dendrogram"), plt.xlabel("Data Points / Clusters")
plt.ylabel("Distance"), plt.grid(True), plt.tight_layout(), plt.show()

# Step 6-8: Agglomerative Clustering and Add Labels
n_clusters = 3
cluster_labels = AgglomerativeClustering(n_clusters=n_clusters, metric="euclidean", linkage="ward").fit_predict(X_scaled)
df["Cluster"] = cluster_labels + 1
print(f"\nClustered Dataset:\n{df.head(10)}")

# Step 9-11: Results, Distribution, Silhouette, and Summary
cluster_counts = df["Cluster"].value_counts().sort_index()
print(f"\n===================================\nHIERARCHICAL CLUSTERING RESULTS\n===================================")
print(f"Number of Clusters: {n_clusters}\n\nCluster Distribution:\n{cluster_counts}")

silhouette = silhouette_score(X_scaled, cluster_labels)
print(f"\nSilhouette Score: {round(silhouette, 4)}")

cluster_summary = df.groupby("Cluster")[feature_names].mean()
print(f"\nCluster Summary:\n{cluster_summary}")

# Step 12: Visualize Clusters
plt.figure(figsize=(9, 6))
for cluster in range(1, n_clusters + 1):
    pts = df["Cluster"] == cluster
    plt.scatter(X[pts, 0], X[pts, 1], label=f"Cluster {cluster}")
plt.xlabel("Sepal Length (cm)"), plt.ylabel("Sepal Width (cm)")
plt.title("Hierarchical Clustering - Iris Dataset"), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 13-15: Comparison, Crosstab, and Save Results
comparison = pd.DataFrame({"Actual Class": [target_names[i] for i in target], "Cluster": cluster_labels + 1})
print(f"\nActual Class vs Cluster:\n{comparison.head(20)}")

crosstab = pd.crosstab(comparison["Actual Class"], comparison["Cluster"])
print(f"\nActual Class vs Cluster Table:\n{crosstab}")

df.to_csv("Iris_Hierarchical_Clustering_Results.csv", index=False)
print("\nResults saved as:\nIris_Hierarchical_Clustering_Results.csv")